# Customer Churn Prediction — Machine Learning Modeling

This notebook extends the **Customer Churn & Retention Analytics** project from descriptive analysis into predictive modeling.

The goal is to build a classification model that estimates which customers are most likely to churn, compare different modeling approaches, select an appropriate decision threshold, and translate the predictions into actionable customer risk segments.

**Target:** `Churn` — 0 = retained, 1 = churned  
**Main evaluation focus:** precision, recall, F1-score, and ROC-AUC, with particular attention to identifying customers at risk of churn.

## 1. Setup and Load the Processed Dataset

We begin with the cleaned customer dataset produced during the earlier data-preparation and exploratory-analysis work.

The notebook loads the processed CSV from the project's `data/processed` folder so the modeling workflow remains connected to the existing project structure.

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

base = Path.cwd().resolve()
if base.name == "notebooks":
    base = base.parent

csv_path = base / "data" / "processed" / "ecommerce_churn_cleaned.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"File not found: {csv_path}")

df = pd.read_csv(csv_path)

In [3]:
df.head()


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,...,Tenure_Missing,WarehouseToHome_Missing,HourSpendOnApp_Missing,OrderCount_Missing,CouponUsed_Missing,OrderAmountHike_Missing,Tenure_Bucket,Recency_Bucket,Order_Frequency_Bucket,Risk_Score
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,...,0,0,0,0,0,0,0-6mo,3-7 days,Low (1-2),2
1,50002,1,9.0,Mobile Phone,1,8.0,UPI,Male,3.0,4,...,1,0,0,0,0,0,6-12mo,0-2 days,Low (1-2),2
2,50003,1,9.0,Mobile Phone,1,30.0,Debit Card,Male,2.0,4,...,1,0,0,0,0,0,6-12mo,3-7 days,Low (1-2),2
3,50004,1,0.0,Mobile Phone,3,15.0,Debit Card,Male,2.0,4,...,0,0,0,0,0,0,0-6mo,3-7 days,Low (1-2),1
4,50005,1,0.0,Mobile Phone,1,12.0,Credit Card,Male,3.0,3,...,0,0,1,0,0,0,0-6mo,3-7 days,Low (1-2),2


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5630 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5630 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5630 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

## 2. Define the Prediction Target and Modeling Features

The target variable is `Churn`.

Several columns are excluded before modeling:
- `CustomerID` is an identifier rather than a predictive feature.
- `Churn` is the target itself.
- `Tenure_Bucket`, `Recency_Bucket`, and `Order_Frequency_Bucket` are derived versions of numeric variables that are already retained.
- `Risk_Score` is excluded because it was manually constructed from churn-related information and could make the model harder to interpret.

The original numeric variables are therefore used directly wherever possible.

In [5]:
drop_columns = [
    'CustomerID',
    'Churn',
    'Tenure_Bucket',
    'Recency_Bucket',
    'Order_Frequency_Bucket',
    'Risk_Score'
]

In [6]:
y = df['Churn']

In [7]:
X = df.drop(columns=drop_columns)

In [ ]:
X.info()

In [ ]:
X.isnull().sum()

In [10]:
y.value_counts(normalize=True)

Churn
0    0.831616
1    0.168384
Name: proportion, dtype: float64

In [11]:
df[[
    'Tenure_Missing',
    'WarehouseToHome_Missing',
    'HourSpendOnApp_Missing',
    'OrderCount_Missing',
    'CouponUsed_Missing',
    'OrderAmountHike_Missing'
]].sum()

Tenure_Missing             264
WarehouseToHome_Missing    251
HourSpendOnApp_Missing     255
OrderCount_Missing         258
CouponUsed_Missing         256
OrderAmountHike_Missing    265
dtype: int64

## 3. Train-Test Split and Class Balance

The data is split into training and test sets using stratification so that the churn proportion remains similar in both sets.

This is important because churn is an imbalanced target: retained customers are substantially more common than churned customers. The test set is kept separate for final evaluation.

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining churn rate:")
print(y_train.mean())

print("\nTest churn rate:")
print(y_test.mean())

Training set: (4504, 24)
Test set: (1126, 24)

Training churn rate:
0.16829484902309058

Test churn rate:
0.16873889875666073


## 4. Prepare Numeric and Categorical Features

The dataset contains both numerical and categorical variables.

For Logistic Regression:
- numerical features are standardized;
- categorical features are one-hot encoded;
- the transformations are kept inside a pipeline to avoid applying preprocessing outside the model workflow.

In [13]:
categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numeric_features)

Categorical features:
['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']

Numerical features:
['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount', 'Tenure_Missing', 'WarehouseToHome_Missing', 'HourSpendOnApp_Missing', 'OrderCount_Missing', 'CouponUsed_Missing', 'OrderAmountHike_Missing']


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ]
)

## 5. Logistic Regression — Baseline Model

Logistic Regression provides a simple and interpretable baseline for the binary churn-prediction problem.

The model first establishes how well a linear classification approach performs before moving to tree-based models.

In [16]:
logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [17]:
y_pred = logistic_model.predict(X_test)

y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("First 10 predictions:")
print(y_pred[:10])

print("\nFirst 10 churn probabilities:")
print(y_prob[:10])

First 10 predictions:
[0 0 0 0 1 0 0 0 0 0]

First 10 churn probabilities:
[2.17267663e-03 1.09868289e-02 7.51780310e-03 3.06262303e-03
 5.59465739e-01 4.65250992e-04 4.07626033e-02 1.22047580e-03
 1.78251034e-01 1.52698183e-03]


In [18]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[900  36]
 [ 90 100]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.96      0.93       936
           1       0.74      0.53      0.61       190

    accuracy                           0.89      1126
   macro avg       0.82      0.74      0.77      1126
weighted avg       0.88      0.89      0.88      1126

ROC-AUC: 0.8938708951866846


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(f"Threshold: {threshold}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1-score:  {f1:.2f}")
    print("-" * 30)

Threshold: 0.3
Precision: 0.58
Recall:    0.76
F1-score:  0.66
------------------------------
Threshold: 0.4
Precision: 0.64
Recall:    0.63
F1-score:  0.64
------------------------------
Threshold: 0.5
Precision: 0.74
Recall:    0.53
F1-score:  0.61
------------------------------
Threshold: 0.6
Precision: 0.79
Recall:    0.45
F1-score:  0.57
------------------------------
Threshold: 0.7
Precision: 0.88
Recall:    0.30
F1-score:  0.45
------------------------------


## 6. Decision Tree — Non-Linear Baseline

A Decision Tree can capture non-linear relationships and interactions that Logistic Regression may miss.

The tree is trained with balanced class weights so that the minority churn class receives more attention during learning.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline(
    steps=[
        ('preprocessor', tree_preprocessor),
        ('model', DecisionTreeClassifier(
            random_state=42,
            class_weight='balanced'
        ))
    ]
)

In [22]:
tree_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [23]:
y_pred_tree = tree_model.predict(X_test)

y_prob_tree = tree_model.predict_proba(X_test)[:, 1]

In [24]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tree))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_tree))

Confusion Matrix:
[[919  17]
 [ 22 168]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       936
           1       0.91      0.88      0.90       190

    accuracy                           0.97      1126
   macro avg       0.94      0.93      0.94      1126
weighted avg       0.97      0.97      0.97      1126

ROC-AUC: 0.9330240665766981


In [25]:
tree = tree_model.named_steps['model']

print("Tree depth:", tree.get_depth())
print("Number of leaves:", tree.get_n_leaves())

Tree depth: 20
Number of leaves: 379


### Checking for Decision Tree Overfitting

The unrestricted tree is also evaluated on the training data.

A large gap between training and test performance would indicate that the tree is learning the training data too closely rather than generalizing well to unseen customers.

In [26]:
y_pred_tree_train = tree_model.predict(X_train)

print("Training Classification Report:")
print(classification_report(y_train, y_pred_tree_train))

Training Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3746
           1       1.00      1.00      1.00       758

    accuracy                           1.00      4504
   macro avg       1.00      1.00      1.00      4504
weighted avg       1.00      1.00      1.00      4504



In [27]:
y_prob_tree_train = tree_model.predict_proba(X_train)[:, 1]

print("Training ROC-AUC:",
      roc_auc_score(y_train, y_prob_tree_train))

Training ROC-AUC: 1.0


### Regularizing Tree Depth

Maximum tree depth is varied to see how restricting complexity affects generalization.

The comparison is used to understand the trade-off between fitting the training data and maintaining useful performance on unseen customers.

In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

depths = [3, 5, 7, 10]

results = []

for depth in depths:

    model = Pipeline(
        steps=[
            ('preprocessor', tree_preprocessor),
            ('model', DecisionTreeClassifier(
                max_depth=depth,
                random_state=42,
                class_weight='balanced'
            ))
        ]
    )

    model.fit(X_train, y_train)

    # Training predictions
    train_pred = model.predict(X_train)
    train_prob = model.predict_proba(X_train)[:, 1]

    # Test predictions
    test_pred = model.predict(X_test)
    test_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'max_depth': depth,
        'train_accuracy': accuracy_score(y_train, train_pred),
        'test_accuracy': accuracy_score(y_test, test_pred),
        'train_recall': recall_score(y_train, train_pred),
        'test_recall': recall_score(y_test, test_pred),
        'train_f1': f1_score(y_train, train_pred),
        'test_f1': f1_score(y_test, test_pred),
        'train_auc': roc_auc_score(y_train, train_prob),
        'test_auc': roc_auc_score(y_test, test_prob)
    })

import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.round(3))

   max_depth  train_accuracy  test_accuracy  train_recall  test_recall  \
0          3           0.815          0.821         0.755        0.716   
1          5           0.807          0.786         0.872        0.837   
2          7           0.875          0.850         0.902        0.868   
3         10           0.944          0.901         0.962        0.863   

   train_f1  test_f1  train_auc  test_auc  
0     0.579    0.575      0.854     0.844  
1     0.603    0.569      0.901     0.877  
2     0.708    0.661      0.944     0.900  
3     0.852    0.747      0.987     0.909  


## 7. Random Forest — Ensemble Model

Random Forest combines many decision trees to improve generalization and capture more complex patterns.

The baseline configuration uses 200 trees, balanced class weights, and parallel processing.

In [29]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        ('preprocessor', tree_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        ))
    ]
)

In [30]:
random_forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [31]:
y_pred_rf = random_forest_model.predict(X_test)
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Confusion Matrix:
[[935   1]
 [ 25 165]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       936
           1       0.99      0.87      0.93       190

    accuracy                           0.98      1126
   macro avg       0.98      0.93      0.96      1126
weighted avg       0.98      0.98      0.98      1126

ROC-AUC: 0.9985773729194782


In [32]:
rf_model = random_forest_model.named_steps['model']
rf_preprocessor = random_forest_model.named_steps['preprocessor']

feature_names = rf_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

importance_df.head(15)

,Feature,Importance
17,remainder__Tenure,0.237194
29,remainder__CashbackAmount,0.089707
19,remainder__WarehouseToHome,0.062377
24,remainder__Complain,0.062040
28,remainder__DaySinceLastOrder,0.056157
23,remainder__NumberOfAddress,0.052181
25,remainder__OrderAmountHikeFromlastYear,0.048762
22,remainder__SatisfactionScore,0.046796
21,remainder__NumberOfDeviceRegistered,0.030729
12,cat__PreferedOrderCat_Mobile Phone,0.030562


## 8. Churn Pattern Checks and Feature Interpretation

Model performance alone is not enough for a business analytics project.

These checks compare important customer characteristics between retained and churned customers. They help connect the predictive model back to the earlier descriptive analysis and provide context for the model's predictions.

Feature importance is treated as **predictive usefulness, not causality**.

In [33]:
analysis_df = X.copy()
analysis_df['Churn'] = y

In [34]:
analysis_df.groupby('Churn')['Tenure'].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
0,4682.0,11.404528,8.267244,0.0,5.0,10.0,17.0,61.0
1,948.0,3.859705,5.476680,0.0,0.0,1.0,7.0,21.0


In [35]:
analysis_df.groupby('Churn')['CashbackAmount'].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
0,4682.0,180.635203,50.434791,0.00,147.24,166.115,201.25,324.99
1,948.0,160.370928,38.436055,110.09,132.02,149.660,174.69,323.59


In [36]:
analysis_df.groupby('Churn')['DaySinceLastOrder'].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
0,4682.0,4.709739,3.568374,0.0,2.0,3.0,8.0,31.0
1,948.0,3.222574,3.316790,0.0,1.0,2.5,4.0,46.0


In [37]:
analysis_df.groupby('Churn')[[
    'Complain',
    'SatisfactionScore',
    'OrderCount',
    'NumberOfDeviceRegistered',
    'OrderAmountHikeFromlastYear'
]].mean()

,Complain,SatisfactionScore,OrderCount,NumberOfDeviceRegistered,OrderAmountHikeFromlastYear
Churn,,,,,
0,0.234088,3.001282,2.992952,3.639257,15.686032
1,0.535865,3.390295,2.808017,3.934599,15.618143


In [38]:
analysis_df.groupby('Churn')['Complain'].value_counts(normalize=True)

Churn  Complain
0      0           0.765912
       1           0.234088
1      1           0.535865
       0           0.464135
Name: proportion, dtype: float64

In [39]:
analysis_df.groupby('Churn').mean(numeric_only=True).T.sort_values(
    by=1, ascending=False
)

Churn,0,1
CashbackAmount,180.635203,160.370928
WarehouseToHome,15.268689,16.856540
OrderAmountHikeFromlastYear,15.686032,15.618143
NumberOfAddress,4.162965,4.466245
NumberOfDeviceRegistered,3.639257,3.934599
Tenure,11.404528,3.859705
SatisfactionScore,3.001282,3.390295
DaySinceLastOrder,4.709739,3.222574
HourSpendOnApp,2.928663,2.964135
OrderCount,2.992952,2.808017


## 9. Cross-Validation and Model Comparison

A single train-test split can give an incomplete view of model performance.

Five-fold stratified cross-validation is used on the training data to compare Logistic Regression, Decision Tree, and Random Forest across accuracy, precision, recall, F1-score, and ROC-AUC.

The test set remains reserved for the final evaluation.

In [40]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_cv = cross_validate(
    random_forest_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
},
    n_jobs=-1
)

print("Mean CV Accuracy:", rf_cv['test_accuracy'].mean())
print("Mean CV Precision:", rf_cv['test_precision'].mean())
print("Mean CV Recall:", rf_cv['test_recall'].mean())
print("Mean CV F1:", rf_cv['test_f1'].mean())
print("Mean CV ROC-AUC:", rf_cv['test_roc_auc'].mean())

Mean CV Accuracy: 0.9458248859292144
Mean CV Precision: 0.9431128516716101
Mean CV Recall: 0.7216364586964099
Mean CV F1: 0.8168692188660301
Mean CV ROC-AUC: 0.9797458473968543


In [41]:
models = {
    'Logistic Regression': logistic_model,
    'Decision Tree': tree_model,
    'Random Forest': random_forest_model
}

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            'accuracy': 'accuracy',
            'precision': 'precision',
            'recall': 'recall',
            'f1': 'f1',
            'roc_auc': 'roc_auc'
        },
        n_jobs=-1
    )

    cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean()
    })

comparison_df = pd.DataFrame(cv_results)

comparison_df.sort_values('ROC-AUC', ascending=False)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
2,Random Forest,0.945825,0.943113,0.721636,0.816869,0.979746
0,Logistic Regression,0.894538,0.764645,0.539587,0.631742,0.896181
1,Decision Tree,0.924735,0.780404,0.770521,0.775186,0.863236


## 10. Random Forest Hyperparameter Tuning

After comparing the baseline models, Random Forest is tuned using `RandomizedSearchCV`.

The search focuses on tree count, tree depth, split and leaf sizes, and the number of features considered at each split. ROC-AUC is used as the tuning objective.

The tuned model is then compared against the baseline rather than being assumed to be better simply because it was tuned.

In [42]:
from sklearn.model_selection import RandomizedSearchCV

rf_pipeline = random_forest_model

param_grid = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [None, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation s

In [43]:
print("Best parameters:")
print(rf_search.best_params_)

print("\nBest CV ROC-AUC:")
print(rf_search.best_score_)

Best parameters:
{'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': None}

Best CV ROC-AUC:
0.9747143845254034


In [44]:
tuned_rf = rf_search.best_estimator_

y_pred_tuned = tuned_rf.predict(X_test)
y_prob_tuned = tuned_rf.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

Confusion Matrix:
[[925  11]
 [ 24 166]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       936
           1       0.94      0.87      0.90       190

    accuracy                           0.97      1126
   macro avg       0.96      0.93      0.94      1126
weighted avg       0.97      0.97      0.97      1126

ROC-AUC: 0.9958277103013946


## 11. Selecting a Churn Decision Threshold

A model produces a churn probability, but the business still needs a cutoff for converting that probability into a churn flag.

An initial threshold table is shown on the held-out test set for exploration. The final threshold is **not selected from this table**; instead, out-of-fold training predictions are used to choose the threshold without repeatedly optimizing against the test set.

In [45]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    threshold_results.append({
        'Threshold': threshold,
        'Precision': precision_score(y_test, y_pred_threshold),
        'Recall': recall_score(y_test, y_pred_threshold),
        'F1': f1_score(y_test, y_pred_threshold)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1
0,0.2,0.756972,1.000000,0.861678
1,0.3,0.882075,0.984211,0.930348
2,0.4,0.962766,0.952632,0.957672
3,0.5,0.993976,0.868421,0.926966
4,0.6,1.000000,0.757895,0.862275
5,0.7,1.000000,0.573684,0.729097
6,0.8,1.000000,0.294737,0.455285


In [46]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_prob_rf = cross_val_predict(
    random_forest_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

print(oof_prob_rf[:10])

[0.23  0.005 0.05  0.81  0.01  0.045 0.08  0.015 0.03  0.15 ]


In [47]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

cv_threshold_results = []

for t in thresholds:
    y_pred = (oof_prob_rf >= t).astype(int)

    cv_threshold_results.append({
        'Threshold': t,
        'Precision': precision_score(y_train, y_pred),
        'Recall': recall_score(y_train, y_pred),
        'F1': f1_score(y_train, y_pred)
    })

cv_threshold_df = pd.DataFrame(cv_threshold_results)

cv_threshold_df

,Threshold,Precision,Recall,F1
0,0.2,0.709742,0.941953,0.809524
1,0.3,0.842105,0.886544,0.863753
2,0.4,0.904070,0.820580,0.860304
3,0.5,0.943493,0.726913,0.821162
4,0.6,0.968220,0.602902,0.743089
5,0.7,0.991018,0.436675,0.606227
6,0.8,0.994819,0.253298,0.403785


### Final Threshold and Held-Out Test Evaluation

A threshold of **0.30** is selected from the out-of-fold results because it gives the strongest F1-score among the tested thresholds while maintaining high recall.

The Random Forest is then fitted on the full training set and evaluated once on the untouched test set using this threshold.

In [ ]:
selected_threshold = 0.30

random_forest_model.fit(X_train, y_train)

y_prob_final = random_forest_model.predict_proba(X_test)[:, 1]

y_pred_final = (y_prob_final >= selected_threshold).astype(int)

print("Final Threshold:", selected_threshold)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_prob_final))

Final Threshold: 0.3

Confusion Matrix:
[[911  25]
 [  3 187]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98       936
           1       0.88      0.98      0.93       190

    accuracy                           0.98      1126
   macro avg       0.94      0.98      0.96      1126
weighted avg       0.98      0.98      0.98      1126


ROC-AUC:
0.9985773729194782


## 12. Generate Customer-Level Churn Predictions

The final model produces a churn probability for each customer in the test set.

These predictions are linked back to `CustomerID` so the model output can be interpreted at the individual-customer level rather than only as aggregate model metrics.

In [49]:
test_customer_ids = df.loc[X_test.index, 'CustomerID']

print(test_customer_ids.head())
print(test_customer_ids.shape)

4006    54007
1969    51970
4235    54236
105     50106
2295    52296
Name: CustomerID, dtype: int64
(1126,)


In [50]:
print(test_customer_ids.index.equals(X_test.index))

True


In [51]:
prediction_df = pd.DataFrame({
    'CustomerID': test_customer_ids.values,
    'Actual_Churn': y_test.values,
    'Churn_Probability': y_prob_final,
    'Predicted_Churn': y_pred_final
})

prediction_df.head(10)

,CustomerID,Actual_Churn,Churn_Probability,Predicted_Churn
0,54007,0,0.005,0
1,51970,0,0.010,0
2,54236,0,0.050,0
3,50106,0,0.025,0
4,52296,0,0.065,0
5,54394,0,0.005,0
6,52952,0,0.065,0
7,53404,0,0.000,0
8,53586,0,0.060,0
9,50468,0,0.010,0


In [52]:
print(prediction_df.shape)
print(prediction_df['Predicted_Churn'].value_counts())

(1126, 4)
Predicted_Churn
0    914
1    212
Name: count, dtype: int64


In [53]:
high_risk_customers = prediction_df.sort_values(
    'Churn_Probability',
    ascending=False
)

high_risk_customers.head(20)

,CustomerID,Actual_Churn,Churn_Probability,Predicted_Churn
935,54023,1,0.975,1
519,54618,1,0.965,1
404,54288,1,0.960,1
216,55434,1,0.960,1
933,51012,1,0.950,1
193,51803,1,0.945,1
427,50936,1,0.935,1
178,53751,1,0.935,1
775,54870,1,0.930,1
185,51215,1,0.925,1


## 13. Turn Churn Probabilities into Risk Segments

Probability bands provide a more useful business view of the model output than a simple churn / no-churn label.

Customers are first grouped into probability bands and then into three practical risk levels:
- **Low:** below 30%
- **Medium:** 30% to below 50%
- **High:** 50% and above

The observed churn rate within each group is calculated on the held-out test set.

In [54]:
prediction_df['Probability_Band'] = pd.cut(
    prediction_df['Churn_Probability'],
    bins=[0, 0.10, 0.30, 0.50, 0.70, 1.00],
    labels=[
        '0-10%',
        '10-30%',
        '30-50%',
        '50-70%',
        '70-100%',
        ],
    include_lowest=True
)

prediction_df['Probability_Band'].value_counts().sort_index()

Probability_Band
0-10%      791
10-30%     126
30-50%      43
50-70%      58
70-100%    108
Name: count, dtype: int64

In [55]:
band_analysis = (
    prediction_df
    .groupby('Probability_Band', observed=False)
    .agg(
        Customers=('CustomerID', 'count'),
        Actual_Churners=('Actual_Churn', 'sum'),
        Actual_Churn_Rate=('Actual_Churn', 'mean')
    )
    .reset_index()
)

band_analysis['Actual_Churn_Rate'] = (
    band_analysis['Actual_Churn_Rate'] * 100
).round(2)

band_analysis

,Probability_Band,Customers,Actual_Churners,Actual_Churn_Rate
0,0-10%,791,0,0.00
1,10-30%,126,3,2.38
2,30-50%,43,22,51.16
3,50-70%,58,57,98.28
4,70-100%,108,108,100.00


In [56]:
prediction_df['Risk_Level'] = pd.cut(
    prediction_df['Churn_Probability'],
    bins=[-0.001, 0.30, 0.50, 1.00],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

risk_analysis = (
    prediction_df
    .groupby('Risk_Level', observed=False)
    .agg(
        Customers=('CustomerID', 'count'),
        Actual_Churners=('Actual_Churn', 'sum'),
        Actual_Churn_Rate=('Actual_Churn', 'mean')
    )
    .reset_index()
)

risk_analysis['Actual_Churn_Rate'] = (
    risk_analysis['Actual_Churn_Rate'] * 100
).round(2)

risk_analysis

,Risk_Level,Customers,Actual_Churners,Actual_Churn_Rate
0,Low,917,3,0.33
1,Medium,43,22,51.16
2,High,166,165,99.40


## 14. Profile the Customers Identified as High Risk

The final step connects predicted risk back to customer characteristics.

The analysis looks at tenure, complaint history, satisfaction, recent activity, order behavior, cashback, delivery distance, and preferred order category. The purpose is not to claim that these features cause churn, but to understand the type of customer the model is prioritizing for retention.

In [ ]:
risk_customers = df.loc[
    X_test.index,
    [
        'CustomerID',
        'Tenure',
        'Complain',
        'SatisfactionScore',
        'DaySinceLastOrder',
        'OrderCount',
        'CashbackAmount',
        'WarehouseToHome',
        'PreferedOrderCat'
    ]
].copy()

risk_customers['Churn_Probability'] = pd.Series(
    y_prob_final,
    index=X_test.index
)
risk_customers['Risk_Level'] = pd.Series(
    prediction_df['Risk_Level'].values,
    index=X_test.index
)
risk_customers['Actual_Churn'] = pd.Series(
    y_test.values,
    index=X_test.index
)

risk_customers.head()


,CustomerID,Tenure,Complain,SatisfactionScore,DaySinceLastOrder,OrderCount,CashbackAmount,WarehouseToHome,PreferedOrderCat,Churn_Probability,Risk_Level,Actual_Churn
4006,54007,17.0,0,1,8.0,8.0,181.75,9.0,Laptop & Accessory,0.005,Low,0
1969,51970,26.0,1,1,3.0,1.0,209.34,13.0,Fashion,0.010,Low,0
4235,54236,5.0,0,1,9.0,2.0,169.24,15.0,Laptop & Accessory,0.050,Low,0
105,50106,25.0,0,2,2.0,2.0,132.12,8.0,Mobile Phone,0.025,Low,0
2295,52296,4.0,1,3,6.0,9.0,198.52,8.0,Fashion,0.065,Low,0


In [58]:
risk_customers.groupby('Risk_Level', observed=False)[
    [
        'Tenure',
        'Complain',
        'SatisfactionScore',
        'DaySinceLastOrder',
        'OrderCount',
        'CashbackAmount',
        'WarehouseToHome'
    ]
].mean().round(2)

,Tenure,Complain,SatisfactionScore,DaySinceLastOrder,OrderCount,CashbackAmount,WarehouseToHome
Risk_Level,,,,,,,
Low,11.55,0.24,3.01,4.74,3.15,180.34,14.85
Medium,5.09,0.63,3.33,3.47,3.56,171.56,14.88
High,3.73,0.59,3.29,3.25,2.79,157.05,17.16


In [59]:
high_risk = risk_customers[
    risk_customers['Risk_Level'] == 'High'
].copy()

high_risk.shape

(166, 12)

In [ ]:
high_risk[
    [
        'Tenure',
        'Complain',
        'SatisfactionScore',
        'DaySinceLastOrder',
        'OrderCount',
        'CashbackAmount',
        'WarehouseToHome',
        'PreferedOrderCat'
    ]
].describe(include='all')

In [ ]:
high_risk['PreferedOrderCat'].value_counts()

In [62]:
high_risk['Complain'].value_counts(normalize=True)

Complain
1    0.590361
0    0.409639
Name: proportion, dtype: float64

In [63]:
high_risk['SatisfactionScore'].value_counts(normalize=True).sort_index()

SatisfactionScore
1    0.156627
2    0.102410
3    0.295181
4    0.186747
5    0.259036
Name: proportion, dtype: float64

## 15. Business Takeaway

The modeling workflow moves from **prediction to prioritization**.

Customers identified as high risk should receive the strongest retention attention, followed by medium-risk customers. Retention resources do not need to be distributed equally across the entire customer base.

The customer profile analysis can help tailor those interventions, while the effectiveness of any retention action should ultimately be measured through follow-up analysis or controlled experimentation.

The model identifies **who is at risk**; business testing is still needed to determine **which intervention actually reduces churn**.

### Modeling Notes

- Model comparison is based on cross-validation on the training data.
- The final threshold is selected from out-of-fold predictions rather than from the final test set.
- The final test set is used for the held-out performance check.
- Feature importance describes predictive contribution, not causal impact.
- Before treating the model as production-ready, feature timing and potential temporal leakage should be validated against the business's actual churn definition.